# Notebook 1 ? Brownian-Motion Validation

In [ ]:
import sys
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()), pathlib.Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

from stable_baselines3 import PPO

from procs.stochastic_processes import (
    BrownianMotionMidpriceModel,
    ExponentialFillFunction,
    PoissonArrivalModel,
)
from procs.gym.calibration import tune_gamma
from procs.gym.experiment_config import BMExperimentConfig
from procs.gym.helpers.plotting import plot_learned_policy, plot_trajectory
from procs.gym.model_dynamics import LimitOrderModelDynamics
from procs.gym.notebook_support import evaluate_agent_over_seeds, evaluate_as_fast, summarise_agent_frames
from procs.gym.reward_scale import estimate_reward_scale
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.trading_environment import TradingEnvironment
from procs.rewards import CjMmCriterion, PnLReward
from procs.agents import AvellanedaStoikovAgent, Sb3Agent

## 1. Shared Configuration and Environment

In [ ]:
cfg = BMExperimentConfig()
cfg.ensure_artifact_dirs()

reward_scale = estimate_reward_scale(
    sigma=cfg.sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=cfg.terminal_time,
    n_steps=cfg.n_steps,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    num_trajectories=cfg.reward_scale_num_trajectories,
    use_bm=True,
)
print(f"Reward scale: {reward_scale:.6f}")


def get_bm_env(num_trajectories=1, *, normalise=False, use_cj=False):
    reward_fn = CjMmCriterion(per_step_inventory_aversion=cfg.phi) if use_cj else PnLReward()
    return TradingEnvironment(
        model_dynamics=LimitOrderModelDynamics(
            midprice_model=BrownianMotionMidpriceModel(
                volatility=cfg.sigma,
                initial_price=cfg.s0,
                terminal_time=cfg.terminal_time,
                n_steps=cfg.n_steps,
                num_trajectories=num_trajectories,
            ),
            arrival_model=PoissonArrivalModel(
                np.array([cfg.A, cfg.A]),
                num_trajectories,
                use_linear_approximation=True,
            ),
            fill_probability_model=ExponentialFillFunction(cfg.kappa, num_trajectories),
            num_trajectories=num_trajectories,
        ),
        reward_function=reward_fn,
        max_inventory=cfg.q_max,
        normalise_observation_space=normalise,
        normalise_action_space=normalise,
        normalise_rewards=normalise,
        reward_scale=reward_scale if normalise else None,
    )

## 2. Tune A-S Risk Aversion

In [ ]:
dt_bm = cfg.terminal_time / cfg.n_steps
rng = np.random.default_rng(0)
bm_prices = np.zeros(cfg.n_steps + 1)
bm_prices[0] = cfg.s0
for i in range(1, cfg.n_steps + 1):
    bm_prices[i] = bm_prices[i - 1] + cfg.sigma * np.sqrt(dt_bm) * rng.normal()
bm_dt = np.full(cfg.n_steps + 1, dt_bm)
bm_dt[0] = 0.0

as_gamma, study = tune_gamma(
    midprices=bm_prices,
    dt_array=bm_dt,
    sigma=cfg.sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    gamma_range=cfg.as_gamma_range,
    n_trials=cfg.as_gamma_trials,
    num_trajectories=cfg.as_gamma_num_trajectories,
)
print(f"Using A-S ? = {as_gamma:.4f}")

## 3. A-S Baseline

In [ ]:
as_agent = AvellanedaStoikovAgent(as_gamma, cfg.sigma, cfg.kappa, cfg.terminal_time, cfg.tick_size)
plot_trajectory(get_bm_env(1), as_agent, seed=cfg.evaluation_seed)

seed_range = range(cfg.evaluation_seed, cfg.evaluation_seed + cfg.evaluation_rollouts)
as_eval = evaluate_as_fast(
    bm_prices,
    bm_dt,
    gamma=as_gamma,
    sigma=cfg.sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=cfg.terminal_time,
    tick_size=cfg.tick_size,
    q_max=cfg.q_max,
    seeds=seed_range,
    use_linear_approximation=True,
)
print(summarise_agent_frames({"A-S Baseline": as_eval}).to_string(float_format="%.6f"))

## 4. Train PPO

In [ ]:
%%time
train_env = get_bm_env(cfg.n_train, normalise=True, use_cj=True)
sb3_env = StableBaselinesTradingEnvironment(train_env)

model = PPO(
    "MlpPolicy",
    sb3_env,
    tensorboard_log=str(cfg.tensorboard_dir),
    verbose=1,
    device="cpu",
    **cfg.ppo_kwargs(),
)
model.learn(total_timesteps=cfg.ppo_total_timesteps)
model.save(str(cfg.model_path("ppo_bm_unconstrained")))
print(f"Saved PPO model to {cfg.model_path('ppo_bm_unconstrained').with_suffix('.zip')}")

## 5. Evaluate Like-for-Like

In [ ]:
ppo_agent = Sb3Agent(model, train_env=train_env)
plot_learned_policy(ppo_agent, cfg.s0, cfg.terminal_time, cfg.n_steps)

ppo_eval = evaluate_agent_over_seeds(
    lambda: get_bm_env(1, normalise=False, use_cj=False),
    ppo_agent,
    seeds=seed_range,
)
comparison = summarise_agent_frames({
    "A-S Baseline": as_eval,
    "PPO Agent": ppo_eval,
})
comparison.to_csv(cfg.result_path("nb1_bm_comparison_summary.csv"))
print(comparison.to_string(float_format="%.6f"))
print(f"Saved summary to {cfg.result_path('nb1_bm_comparison_summary.csv')}")